# V2 atanmis deneyler — Run All yalniz onayli paketi baslatir; taslak paket baslatilmaz.


In [1]:
import sys, os, json
from pathlib import Path
assert sys.platform == "linux" and "openmc-proje" in sys.executable, "WSL openmc-proje kernelini secin"
PACKAGE = Path.cwd().resolve()
assert PACKAGE.name == "v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1", "Notebook paket dizininden acilmali"
PROJECT = next(p for p in (PACKAGE, *PACKAGE.parents) if (p / "src/reactor_pipeline/v2_students.py").is_file())
assert (PROJECT / "src/reactor_pipeline/v2_students.py").is_file()
sys.path.insert(0, str(PROJECT / "src"))
import openmc
from reactor_pipeline import v2_students, v2_student_runner
print("OpenMC", openmc.__version__, "| veri", os.environ.get("OPENMC_CROSS_SECTIONS"))
assert os.environ.get("OPENMC_CROSS_SECTIONS") and Path(os.environ["OPENMC_CROSS_SECTIONS"]).is_file()
info = v2_students.verify_package(PACKAGE, require_approval=False)
manifest = v2_students._read(PACKAGE / "manifest.json")
print("Grup", manifest["group_id"], "| durum", info["state"])
for case in manifest["experiments"]:
    print(case["experiment_id"], case["scenario_id"], case["controls"], case["calculation"])


OpenMC 0.16.0 | veri /home/ascanipek/openmc-proje/data/endfb-viii.0/endfb-viii.0-hdf5/cross_sections.xml
Grup v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1 | durum production_ready
V2-STAGE_256_OGRENCI_01-0001 29bd0bab7e91ae40ccb8f2bf9a926b87f6fa0736a426f00a72fab572d5e4c464 {'u235_enrichment_wt_percent': 2.264066915769746, 'fuel_temperature_K': 566.0260720196943, 'moderator_temperature_K': 573.0589495420113, 'coolant_pressure_MPa': 15.889761160480582, 'elemental_boron_ppm': 119.10658776603162, 'pin_pitch_cm': 1.2633855437224704, 'pellet_radius_cm': 0.39156405663004634, 'insertion_percent': 76.09718192491152} {'particles_per_generation': 80000, 'batches': 600, 'inactive': 300, 'generations_per_batch': 1, 'threads': 6, 'seed': 71001}
V2-STAGE_256_OGRENCI_01-0002 dac3eee59f2053353688a84623edd83c4cec68cc88c93981b4591292589a5433 {'u235_enrichment_wt_percent': 1.8890669157697462, 'fuel_temperature_K': 572.692738686361, 'moderator_temperature_K': 557.0589495420113, 'coolant_pressure_MPa': 

In [2]:
check = v2_student_runner.preflight(PACKAGE)
print("On kontrol:", check)


On kontrol: {'group_id': 'v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1', 'cases': [{'experiment_id': 'V2-STAGE_256_OGRENCI_01-0001', 'scenario_id': '29bd0bab7e91ae40ccb8f2bf9a926b87f6fa0736a426f00a72fab572d5e4c464', 'geometry_checks': 10, 'water_density_g_cm3': 0.7275274224121423, 'nuclear_data': 'all required library temperatures bracket requested material temperatures', 'calculation': {'particles_per_generation': 80000, 'batches': 600, 'inactive': 300, 'generations_per_batch': 1, 'threads': 6, 'seed': 71001}}, {'experiment_id': 'V2-STAGE_256_OGRENCI_01-0002', 'scenario_id': 'dac3eee59f2053353688a84623edd83c4cec68cc88c93981b4591292589a5433', 'geometry_checks': 10, 'water_density_g_cm3': 0.7568356620321604, 'nuclear_data': 'all required library temperatures bracket requested material temperatures', 'calculation': {'particles_per_generation': 80000, 'batches': 600, 'inactive': 300, 'generations_per_batch': 1, 'threads': 6, 'seed': 71002}}, {'experiment_id': 'V2-STAGE_256_OGRENCI_01-

In [3]:
current = v2_student_runner.status(PACKAGE)
if current["state"] == "not_started":
    v2_students.verify_package(PACKAGE, require_approval=True)
    current = v2_student_runner.launch(PACKAGE)
print("Durum", current["state"], "PID", current.get("pid"))
print("Log", current.get("group_log"), "durum", current.get("status_file"), "sonuclar", current.get("group_dir"))


Durum queued PID 457680
Log /home/ascanipek/openmc-proje/runs/beavrs_3d/v2_groups/v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1/group.log durum /home/ascanipek/openmc-proje/runs/beavrs_3d/v2_groups/v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1/status.json sonuclar /home/ascanipek/openmc-proje/runs/beavrs_3d/v2_groups/v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1


## Teslim — hesap başlatmaz
Tamamlanan koşular için WSL terminalinde: `python scripts/v2_transfer.py export student_packages/v2_students_portable_v1/stage_256/ogrenci_01/v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1 deliveries/v2_stage_256_ogrenci_01_beavrs_v2_pilot_scope_v1.zip`. Durum `completed` olmadan teslim tam sayılmaz; varsayılan ZIP statepoint içermez.
